# USP5 — Latih di Data Lebih Banyak, Uji di 53 Senyawa Mann

**Apa tujuan notebook ini?**
Hasil model dosen turun saat dipakai di 53 senyawa. Notebook ini mencoba tiga hal, dengan aturan yang adil:

1. **Memastikan** apakah 53 senyawa Mann benar-benar ada di dataset 80 (dicek lewat ChEMBL ID dan struktur).
2. **Skenario A vs B.** A: model dilatih hanya dengan senyawa Mann. B: model dilatih dengan semua senyawa (80), tetapi **diuji hanya pada senyawa Mann**. Senyawa uji dan pembagian foldnya sama persis di A dan B, jadi perbandingannya adil.
3. **Fitur lebih ringkas** (model Ridge dengan 15 deskriptor) dan **ukuran tambahan** (Spearman dan pembanding "tebak rata-rata").

**Cara pakai:** jalankan sel dari atas ke bawah. Setiap sel kode didahului penjelasan "Apa ini?".

## 1. Persiapan (import library)

**Apa ini?** Memanggil semua alat yang dibutuhkan: RDKit untuk membaca molekul, scikit-learn untuk model, pandas untuk tabel. Hasil akan disimpan di folder `hasil_skenario_80_uji_53`.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

warnings.filterwarnings('ignore', category=FutureWarning)
OUT = Path('hasil_skenario_80_uji_53')
OUT.mkdir(parents=True, exist_ok=True)
print('RDKit:', rdBase.rdkitVersion, '| folder hasil:', OUT.resolve())

## 2. Memuat dua dataset

**Apa ini?** Mengambil dua file dari GitHub-mu:
- `USP5_pKd_80.csv`: dataset 80 senyawa dari kurasi KNIME (dipakai di notebook dosen).
- `USP5_MANN_53.csv`: 53 senyawa dari artikel Mann (hasil filter `CHEMBL5226343` di KNIME).

Kolom yang wajib ada: `Molecule ChEMBL ID`, `Smiles`, `pKd`. Kalau ada yang hilang, sel ini akan berhenti dan memberi tahu.

In [ ]:
URL_80 = 'https://raw.githubusercontent.com/aliyarahmaazizah/USP5/main/USP5_pKd_80.csv'
URL_53 = 'https://raw.githubusercontent.com/aliyarahmaazizah/USP5/main/USP5_MANN_53.csv'

df80 = pd.read_csv(URL_80)
df53 = pd.read_csv(URL_53)
for d in (df80, df53):
    d.columns = d.columns.str.strip()      # buang spasi liar di nama kolom

wajib = {'Molecule ChEMBL ID', 'Smiles', 'pKd'}
assert wajib.issubset(df80.columns), f'Kolom 80 kurang: {wajib - set(df80.columns)}'
assert wajib.issubset(df53.columns), f'Kolom 53 kurang: {wajib - set(df53.columns)}'
df80['pKd'] = pd.to_numeric(df80['pKd'], errors='raise')
df53['pKd'] = pd.to_numeric(df53['pKd'], errors='raise')
print('Jumlah baris  | dataset 80:', len(df80), '| dataset 53:', len(df53))

## 3. Memastikan 53 senyawa itu ada di dataset 80

**Apa ini?** Ini jawaban untuk keraguanmu. Pengecekan dilakukan dua cara:

1. **Lewat ChEMBL ID**: apakah ID senyawa di 53 juga ada di 80?
2. **Lewat struktur (SMILES kanonik)**: kadang ID berbeda tetapi strukturnya sama, jadi SMILES diseragamkan dulu dengan RDKit.

Sel ini juga membandingkan nilai pKd untuk senyawa yang sama. Kalau ada selisih, akan ditampilkan.

In [ ]:
def kanonik(smiles):
    # Ubah SMILES ke bentuk baku supaya struktur yang sama selalu tertulis sama.
    m = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(m, canonical=True) if m is not None else None

df80['key'] = df80['Smiles'].apply(kanonik)
df53['key'] = df53['Smiles'].apply(kanonik)
assert df80['key'].notna().all() and df53['key'].notna().all(), 'Ada SMILES yang gagal dibaca'

id80, id53 = set(df80['Molecule ChEMBL ID']), set(df53['Molecule ChEMBL ID'])
key80, key53 = set(df80['key']), set(df53['key'])

print('=== Cek lewat ChEMBL ID ===')
print('ID 53 yang ada di 80   :', len(id53 & id80), 'dari', len(id53))
print('=== Cek lewat struktur ===')
print('Struktur 53 yang ada di 80:', len(key53 & key80), 'dari', len(key53))

hilang = df53[~df53['key'].isin(key80)]
if len(hilang) == 0:
    print('\nKESIMPULAN: semua 53 senyawa Mann ADA di dataset 80.')
else:
    print(f'\nKESIMPULAN: {len(hilang)} senyawa Mann TIDAK ada di dataset 80 (tetap dipakai, lihat tabel):')
    display(hilang[['Molecule ChEMBL ID', 'Smiles', 'pKd']])

# Bandingkan pKd untuk senyawa yang sama di dua file
gabung = df53.merge(df80[['key', 'pKd']], on='key', suffixes=('_53', '_80'))
gabung['selisih'] = (gabung['pKd_53'] - gabung['pKd_80']).abs()
print('\nSelisih pKd maksimum pada senyawa yang sama:', round(float(gabung['selisih'].max()), 4) if len(gabung) else 'tidak ada')
if len(gabung) and (gabung['selisih'] > 0.01).any():
    print('Ada senyawa dengan selisih pKd > 0.01, periksa:')
    display(gabung.loc[gabung['selisih'] > 0.01, ['Molecule ChEMBL ID', 'pKd_53', 'pKd_80', 'selisih']])

## 4. Menyusun satu tabel gabungan

**Apa ini?** Membuat satu tabel berisi semua senyawa tanpa duplikat:
- Senyawa Mann (53) memakai nilai pKd dari file 53 dan diberi tanda `is_mann53 = True`.
- Senyawa lain dari dataset 80 ikut sebagai **data tambahan untuk latihan** (`is_mann53 = False`).

Kalau tadi ternyata ada senyawa Mann yang tidak ada di 80, tetap otomatis masuk.

In [ ]:
mann = df53[['Molecule ChEMBL ID', 'Smiles', 'pKd', 'key']].copy()
mann['is_mann53'] = True
tambahan = df80.loc[~df80['key'].isin(key53), ['Molecule ChEMBL ID', 'Smiles', 'pKd', 'key']].copy()
tambahan['is_mann53'] = False

df_all = pd.concat([mann, tambahan], ignore_index=True)
assert not df_all['key'].duplicated().any(), 'Masih ada struktur ganda di tabel gabungan'
print('Total senyawa gabungan :', len(df_all))
print('  - senyawa Mann (uji)  :', int(df_all['is_mann53'].sum()))
print('  - tambahan (latihan)  :', int((~df_all['is_mann53']).sum()))
print('Rentang pKd Mann       :', round(df_all.loc[df_all.is_mann53, 'pKd'].min(), 3), '-', round(df_all.loc[df_all.is_mann53, 'pKd'].max(), 3))
print('Rentang pKd tambahan   :', round(df_all.loc[~df_all.is_mann53, 'pKd'].min(), 3), '-', round(df_all.loc[~df_all.is_mann53, 'pKd'].max(), 3))
print('SD pKd Mann (pembanding RMSE tebak-rata-rata):', round(df_all.loc[df_all.is_mann53, 'pKd'].std(ddof=0), 3))

## 5. Mengubah molekul menjadi angka (fitur) dan membuat scaffold

**Apa ini?** Komputer tidak bisa membaca gambar molekul, jadi molekul diubah menjadi tiga kelompok angka, sama seperti notebook dosen:
- **Deskriptor RDKit**: sifat fisikokimia (berat, logP, TPSA, dan lainnya).
- **Morgan count fingerprint**: potongan struktur di sekitar tiap atom.
- **MACCS keys**: gugus fungsi standar.

Ada juga **scaffold** (kerangka molekul), dipakai untuk membagi data menjadi kelompok yang strukturnya benar-benar berbeda.

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df_all['Smiles']]
groups = np.array([MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols])
y = df_all['pKd'].to_numpy(dtype=float)
is53 = df_all['is_mann53'].to_numpy(dtype=bool)

desc_names = [n for n, _ in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
DESC = np.asarray([calc.CalcDescriptors(m) for m in mols], dtype=float)
DESC[~np.isfinite(DESC)] = np.nan            # nilai tak hingga dianggap kosong; diisi nanti di dalam fold

mgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
MORGAN = np.asarray([mgen.GetCountFingerprintAsNumPy(m) for m in mols], dtype=float)
MACCS = np.asarray([np.asarray(MACCSkeys.GenMACCSKeys(m), dtype=float) for m in mols])

X = np.hstack([DESC, MORGAN, MACCS])
nD, nM = DESC.shape[1], MORGAN.shape[1]
IDX_DESC = np.arange(0, nD)                  # kolom deskriptor saja
IDX_FPMA = np.arange(nD, X.shape[1])         # Morgan + MACCS
IDX_DESCFP = np.arange(0, nD + nM)           # deskriptor + Morgan

print('Bentuk fitur:', X.shape, '| scaffold unik (semua):', len(set(groups)),
      '| scaffold unik (Mann):', len(set(groups[is53])))

## 6. Tiga model yang dibandingkan

**Apa ini?** Mendefinisikan model. Semua hiperparameter **dikunci** (tidak disetel berdasarkan hasil uji), supaya hasilnya jujur.

1. `Ensemble_lama`: persis sama dengan notebook dosen (BayesianRidge, SVR linear, GradientBoosting, lalu dirata-rata).
2. `Ridge_ringkas`: hanya 15 deskriptor yang paling berkorelasi dengan pKd, lalu dilatih dengan Ridge. 15 fitur dipilih **di dalam data latih saja** (tidak mengintip data uji), supaya tidak terjadi kebocoran data.
3. `Gabungan`: rata-rata 50:50 dari model 1 dan 2.

**Mengapa fitur diringkas?** Ada sekitar 1.400 kolom untuk puluhan senyawa. Model mudah menghafal, bukan belajar.

In [ ]:
class BlockSelector(BaseEstimator, TransformerMixin):
    # Mengambil hanya kolom tertentu dari tabel fitur.
    def __init__(self, idx): self.idx = idx
    def fit(self, X, y=None): return self
    def transform(self, X): return np.asarray(X)[:, self.idx]

class TopKCorr(BaseEstimator, TransformerMixin):
    # Memilih k kolom yang paling berkorelasi dengan pKd (hanya memakai data latih).
    def __init__(self, k=15): self.k = k
    def fit(self, X, y=None):
        Xc = X - X.mean(axis=0)
        yc = y - y.mean()
        denom = np.sqrt((Xc ** 2).sum(axis=0)) * np.sqrt((yc ** 2).sum())
        r = np.abs((Xc * yc[:, None]).sum(axis=0) / np.where(denom == 0, np.nan, denom))
        r = np.nan_to_num(r, nan=0.0)
        self.idx_ = np.argsort(-r)[:min(self.k, X.shape[1])]
        return self
    def transform(self, X): return X[:, self.idx_]

class MeanEnsemble(BaseEstimator, RegressorMixin):
    # Melatih beberapa model lalu merata-ratakan prediksinya.
    def __init__(self, members=None): self.members = members
    def fit(self, X, y):
        self.fitted_ = [clone(m).fit(X, y) for m in self.members]
        return self
    def predict(self, X):
        return np.column_stack([np.ravel(m.predict(X)) for m in self.fitted_]).mean(axis=1)

def pipe(idx, est, topk=None):
    # Urutan: pilih kolom -> isi nilai kosong -> buang kolom konstan -> samakan skala -> (pilih top-k) -> model
    steps = [('select', BlockSelector(idx)),
             ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
             ('variance', VarianceThreshold(threshold=0.0)),
             ('scale', StandardScaler())]
    if topk is not None:
        steps.append(('topk', TopKCorr(k=topk)))
    steps.append(('estimator', est))
    return Pipeline(steps)

def model_ensemble_lama():
    return MeanEnsemble(members=[
        pipe(IDX_FPMA, BayesianRidge()),
        pipe(IDX_FPMA, SVR(kernel='linear', C=0.3, epsilon=0.05)),
        pipe(IDX_DESCFP, GradientBoostingRegressor(n_estimators=400, learning_rate=0.05,
                                                   max_depth=3, subsample=0.8, random_state=42)),
    ])

def model_ridge_ringkas():
    return MeanEnsemble(members=[pipe(IDX_DESC, Ridge(alpha=10.0), topk=15)])

def model_gabungan():
    return MeanEnsemble(members=[model_ensemble_lama(), model_ridge_ringkas()])

MODELS = {'Ensemble_lama': model_ensemble_lama,
          'Ridge_ringkas': model_ridge_ringkas,
          'Gabungan': model_gabungan}
print('Model siap:', ', '.join(MODELS))

## 7. Aturan ujian: bagaimana data dibagi

**Apa ini?** Bagian terpenting supaya perbandingan A dan B adil.

- **Senyawa uji selalu senyawa Mann saja**, dibagi 5 fold. Pada protokol *scaffold*, satu scaffold hanya muncul di satu fold, jadi model diuji pada kerangka molekul yang belum pernah dilihat.
- **Skenario A**: model dilatih memakai senyawa Mann yang bukan fold uji.
- **Skenario B**: model dilatih memakai **semua** senyawa (Mann + tambahan) yang bukan fold uji.
- Untuk protokol scaffold, senyawa latih yang scaffold-nya sama dengan senyawa uji **dibuang dari data latih**. Ini mencegah bocor.
- Sebagai pembanding, dihitung juga **"tebak rata-rata"**: kalau model hanya menebak rata-rata pKd latih, berapa RMSE-nya? Model yang bagus harus lebih baik dari itu.

In [ ]:
idx53 = np.where(is53)[0]

def scaffold_fold_ids(idx, seed, n_folds=5):
    # Bagi senyawa per scaffold, dibuat seimbang ukurannya, urutan scaffold diacak per seed.
    rng = np.random.RandomState(seed)
    g = groups[idx]
    unik = np.unique(g)
    rng.shuffle(unik)
    ukuran = {u: int((g == u).sum()) for u in unik}
    urut = sorted(unik, key=lambda u: -ukuran[u])
    beban = np.zeros(n_folds)
    tugas = {}
    for u in urut:
        f = int(np.argmin(beban))
        tugas[u] = f
        beban[f] += ukuran[u]
    return np.array([tugas[x] for x in g])

def buat_fold_uji(kind, seed):
    if kind == 'scaffold':
        fid = scaffold_fold_ids(idx53, seed)
        return [idx53[fid == k] for k in range(5) if (fid == k).any()]
    return [idx53[te] for _, te in KFold(5, shuffle=True, random_state=seed).split(idx53)]

def indeks_latih(skenario, uji, kind):
    pool = idx53 if skenario == 'A_latih_Mann_saja' else np.arange(len(y))
    pool = np.setdiff1d(pool, uji)
    if kind == 'scaffold':
        pool = pool[~np.isin(groups[pool], np.unique(groups[uji]))]
    return pool

def evaluasi(model_fn, skenario, kind, seed):
    pos = {int(i): p for p, i in enumerate(idx53)}
    pred = np.full(len(idx53), np.nan)
    tebak = np.full(len(idx53), np.nan)
    n_latih = []
    for uji in buat_fold_uji(kind, seed):
        latih = indeks_latih(skenario, uji, kind)
        assert set(latih).isdisjoint(set(uji)), 'Data uji bocor ke data latih'
        if kind == 'scaffold':
            assert set(groups[latih]).isdisjoint(set(groups[uji])), 'Scaffold bocor'
        model = model_fn().fit(X[latih], y[latih])
        p = model.predict(X[uji])
        for i, v in zip(uji, p):
            pred[pos[int(i)]] = v
            tebak[pos[int(i)]] = y[latih].mean()
        n_latih.append(len(latih))
    assert np.isfinite(pred).all()
    yt = y[idx53]
    hasil = dict(R2=r2_score(yt, pred),
                 RMSE=float(np.sqrt(mean_squared_error(yt, pred))),
                 MAE=mean_absolute_error(yt, pred),
                 Spearman=float(spearmanr(yt, pred).correlation),
                 RMSE_tebak_rata2=float(np.sqrt(mean_squared_error(yt, tebak))),
                 rata2_n_latih=float(np.mean(n_latih)))
    return hasil, pred

print('Senyawa uji (Mann):', len(idx53), '| aturan ujian siap')

## 8. Menjalankan semua percobaan

**Apa ini?** Menjalankan setiap kombinasi: 2 protokol (scaffold dan acak) × 2 skenario (A dan B) × 3 model × 5 seed (pengacakan berbeda). Total 60 kombinasi, masing-masing 5 fold. Mungkin butuh beberapa menit. Hasil per seed disimpan supaya bisa diperiksa.

In [ ]:
SEEDS = [42, 123, 2026, 7, 99]
SKENARIO = ['A_latih_Mann_saja', 'B_latih_semua_uji_Mann']

baris, oof = [], {}
for kind in ['scaffold', 'random']:
    for sk in SKENARIO:
        for nama, fn in MODELS.items():
            for seed in SEEDS:
                hasil, pred = evaluasi(fn, sk, kind, seed)
                baris.append(dict(protokol=kind, skenario=sk, model=nama, seed=seed, **hasil))
                oof[(kind, sk, nama, seed)] = pred
            print(f'selesai: {kind:8s} | {sk:24s} | {nama}')

hasil_seed = pd.DataFrame(baris)
hasil_seed.to_csv(OUT / 'hasil_per_seed.csv', index=False)
print('\nTersimpan:', OUT / 'hasil_per_seed.csv')

## 9. Ringkasan hasil

**Apa ini?** Merata-ratakan lima seed menjadi satu tabel (rata-rata dan simpangan baku). Cara membacanya:

- **R² dan Spearman makin tinggi makin baik.** Spearman mengukur ketepatan *urutan* senyawa, dan itu yang paling penting untuk memilih kandidat.
- **RMSE makin kecil makin baik**, dan harus lebih kecil dari `RMSE_tebak_rata2`. Kalau tidak, model tidak lebih pintar dari menebak rata-rata.
- Bandingkan **A vs B** pada model yang sama: kalau B lebih baik, data tambahan membantu.
- Perhatikan kolom `std`. Kalau selisih A dan B lebih kecil dari `std`, perbedaannya belum bisa dipercaya.

In [ ]:
ringkas = (hasil_seed
           .groupby(['protokol', 'skenario', 'model'])[['R2', 'RMSE', 'Spearman', 'RMSE_tebak_rata2', 'rata2_n_latih']]
           .agg(['mean', 'std']).round(3))
ringkas.to_csv(OUT / 'ringkasan.csv')

print('=== PROTOKOL SCAFFOLD (paling relevan: senyawa dengan kerangka baru) ===')
display(ringkas.loc['scaffold'])
print('=== PROTOKOL ACAK (referensi) ===')
display(ringkas.loc['random'])

## 10. Grafik perbandingan

**Apa ini?** Dua grafik batang untuk protokol scaffold: R² dan Spearman untuk setiap model, dipisah antara skenario A dan B. Garis kecil di atas batang adalah simpangan baku antar seed.

In [ ]:
sc = hasil_seed[hasil_seed.protokol == 'scaffold']
agg = sc.groupby(['model', 'skenario'])[['R2', 'Spearman']].agg(['mean', 'std'])
nama_model = list(MODELS)
lebar = 0.38
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, metrik in zip(ax, ['R2', 'Spearman']):
    for j, sk in enumerate(SKENARIO):
        m = [agg.loc[(n, sk), (metrik, 'mean')] for n in nama_model]
        s = [agg.loc[(n, sk), (metrik, 'std')] for n in nama_model]
        a.bar(np.arange(len(nama_model)) + (j - 0.5) * lebar, m, lebar, yerr=s, capsize=3,
              label=sk.replace('_', ' '))
    a.axhline(0, color='k', lw=0.8)
    a.set_xticks(np.arange(len(nama_model)))
    a.set_xticklabels(nama_model, fontsize=8)
    a.set_title(f'{metrik} (scaffold CV, rata-rata 5 seed)')
    a.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT / 'grafik_perbandingan_scaffold.png', dpi=200, bbox_inches='tight')
plt.show()

## 11. Aturan jujur saat melaporkan

Supaya hasilnya aman dipertanggungjawabkan di skripsi:

1. **Laporkan semua kombinasi**, bukan hanya yang paling bagus. Hasil yang kurang bagus tetap temuan.
2. **Jangan mengubah hiperparameter setelah melihat hasil uji.** Kalau ingin mencoba pengaturan lain, jalankan sebagai percobaan baru dan catat.
3. **Ini hasil validasi silang, bukan uji independen.** Tulis sebagai "repeated scaffold cross-validation".
4. **Dataset tambahan (skenario B) mungkin berasal dari assay atau konstruksi protein yang berbeda.** Catat ini sebagai keterbatasan.
5. **Model hanya berlaku untuk senyawa yang mirip data latih.** Untuk struktur yang sangat berbeda (misalnya stilben), hasil prediksinya hanya perkiraan kasar.